# TOML - Rust

All 5 Rust examples from [docs/toml.md](https://platob.github.io/yggdryl/toml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Raw shared-Scalar access

In [ ]:
use yggdryl::{toml, Scalar};

let value = toml::from_utf8(
    "title = \"yggdryl\"\ncount = 3\n\n[owner]\nname = \"Ada\"\n"
)?;
let encoded = toml::into_utf8(&value)?;

assert_eq!(
    value.get_key_str("title").and_then(Scalar::as_utf8),
    Some("yggdryl")
);
assert_eq!(toml::from_utf8(&encoded)?, value);

## Natural values and exact Fields

In [ ]:
use yggdryl::{toml, DataType, Field, Scalar};

let amount = Field::new("amount", DataType::decimal128(8, 2)?, false);
let row = Field::new(
    "row",
    DataType::from_fields([amount])?,
    false,
);
let decoded = toml::from_utf8_with_field("amount = '12.50'\n", &row)?;

assert_eq!(decoded.as_sequence().unwrap()[0], Scalar::d128(1_250, 2));

## Documents and streams

In [ ]:
use yggdryl::toml;

let value = toml::from_utf8("id = 1\n")?;
let mut destination = Vec::new();
toml::into_writer(&value, &mut destination)?;

assert_eq!(toml::from_bytes(&destination)?, value);

## Formatting

In [ ]:
use yggdryl::text::Formatting;
use yggdryl::{toml, Scalar};

let value = Scalar::from_record([(
    "items",
    Scalar::from_sequence([Scalar::I64(1), Scalar::I64(2), Scalar::I64(3)]),
)])?;
let laid_out =
    toml::into_utf8_with_formatting(&value, Formatting::indented(2))?;
let compact = toml::into_utf8_with_formatting(&value, Formatting::compact())?;

assert_ne!(laid_out, compact);
assert_eq!(toml::from_utf8(&laid_out)?, value);

## Placeholders

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Scalar;

let loading = Loading::new().with_placeholders(
    Placeholders::new()
        .with_variable("HOST", Scalar::from("db.internal"))
        .with_variable("PORT", Scalar::I64(5432)),
);
let value = yggdryl::text::from_utf8_with(
    "host = \"{{ HOST }}\"\nport = \"{{ PORT }}\"\n",
    Format::Toml,
    &loading,
)?;

assert_eq!(value.get_key_str("host").and_then(Scalar::as_utf8), Some("db.internal"));
assert_eq!(value.get_key_str("port"), Some(&Scalar::I64(5432)));